# Design and adapt a new LoRaWAN study
**Objective:** produce an executable research specification and an audit before a new model search.
This notebook is a template with an illustrative completed specification, not a new empirical benchmark.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG
ROOT = Path.cwd() if (Path.cwd() / 'data/manifest.json').exists() else Path.cwd().parent
assert (ROOT / 'data/manifest.json').exists(), 'Run from the project root or notebooks directory.'
sys.path.insert(0, str(ROOT / 'scripts'))
PILOT = ROOT / 'results/pilot_v1'
MATCHED = ROOT / 'results/transition_v2'
GUIDE = ROOT / 'results/guide_v3'


## 1. State the prediction contract
Replace each field for your own capture. A valid study can conclude that the available data do not support the intended target.

In [ ]:
contract={
    'question':'Predict the mean received SNR of one future hour at +6 h',
    'unit':'device-gateway-hour',
    'origin':'closure of the latest observed hour',
    'inputs':['24 closed RSSI hours','24 closed SNR hours'],
    'target':'SNR of one hourly interval closing six hours later',
    'label_available_at':'origin + 6 hours',
    'population':'later eligible hours of the same observed link',
    'split':'chronological; labels available before next phase',
    'baseline':['persistence','24-hour mean'],
    'models':['Ridge','RF','MLP','temporal CNN'],
    'selection':'validation MAE; frozen fitting and transformations',
    'metric':'MAE and RMSE in dB; link-level reporting',
    'limits':['received target and complete history required','not delivery probability','not a network intervention']
}
display(pd.Series(contract))
assert all(contract.values())


## 2. Translate the contract into audit conditions

In [ ]:
a=np.load(MATCHED/'uva_sensor01_snr_h6/examples.npz',allow_pickle=False)
s=pd.read_csv(MATCHED/'uva_sensor01_snr_h6/splits.csv.gz')
s['origin']=pd.to_datetime(s.origin,utc=True);s['label_available_at']=pd.to_datetime(s.label_available_at,utc=True)
meta=json.loads((MATCHED/'uva_sensor01_snr_h6/case.json').read_text())
assert s.loc[s.split.eq('train'),'label_available_at'].max()<pd.Timestamp(meta['validation_start'])
assert s.loc[s.split.eq('validation'),'label_available_at'].max()<pd.Timestamp(meta['test_start'])
assert np.isfinite(a['X']).all() and np.isfinite(a['y']).all()
assert len(np.unique(a['row_id']))==len(a['row_id'])
print('Availability, unique IDs and numeric eligibility checks passed.')


## 3. Read a paper as an experimental specification
Extract its observation unit, population, dataset version, input columns, target, partitions, transformations, tuning budget and metric. Write `not reported` when the source does not establish an item. Separate an author statement from your interpretation.

For example, [Anagnostopoulos and Kalousis (2019), Sections III–VI](https://arxiv.org/html/1908.05085v1) compare RSSI fingerprint learners, but Extra Trees and MLP also receive spreading factor. Their random partitions and our temporal/spatial examples support different questions. A lower error in one table cannot isolate an architectural improvement across these protocols.

**Exercise:** fill two columns, `published contract` and `proposed contract`, for a paper relevant to your research. Mark each changed field. Classify your experiment as a reproduction, an information-matched model comparison, or an extension to a new question/population. Explain which numerical comparisons remain meaningful.

## 4. Connect a probability to an explicit decision
This is illustrative arithmetic, not a fitted classifier or an observed network benefit. Let the event be a future low-SNR condition and the action be flagging it for review. Assume a calibrated event probability, false-positive cost 1, false-negative cost 4, and zero cost for correct decisions. Then flagging costs `(1-p)*c_FP` in expectation; not flagging costs `p*c_FN`. The threshold is `c_FP/(c_FP+c_FN)`. Here a tie is resolved by not flagging.

In [ ]:
probability = 0.30
false_positive_cost, false_negative_cost = 1., 4.
threshold = false_positive_cost / (false_positive_cost + false_negative_cost)
expected_cost = pd.Series({
    'flag for review': (1 - probability) * false_positive_cost,
    'do not flag': probability * false_negative_cost,
})
flag = expected_cost['flag for review'] < expected_cost['do not flag']
np.testing.assert_allclose(expected_cost.to_numpy(), [0.7, 1.2])
assert flag == (probability > threshold)
display(expected_cost.rename('illustrative expected cost'))
print(f'Threshold: {threshold:.2f}; event probability: {probability:.2f}; flag: {flag}')


**Exercise:** increase the false-positive cost from 1 to 8. Recompute both expected costs and the decision. Why might the threshold fail after moving to another deployment?

<details><summary>Feedback after your attempt</summary>

the threshold becomes 8/12, flagging costs 5.6 and not flagging costs 1.2; do not flag. Transfer can change calibration or costs, and limited review capacity can require a different policy. Changing event prevalence is a reason to reassess probabilities, not proof that an old threshold remains valid. If an action changes the network outcome, these observational probabilities do not identify the intervention's effect. Estimating energy or delivery benefits requires a separate policy evaluation.

</details>

## 5. Plan one discriminating experiment
Choose one question: Does a new covariate help? Does a representation help? Does adding independent collection units help? Does a model transfer to a new receiver? State what is held fixed and what changes. A test already inspected during development is not a fresh confirmatory holdout.

## 6. Report what was observed and what remains a hypothesis
A report should contain data provenance; units and availability; exclusions; partitions; preprocessing; search space and budget; selected settings; prediction-level metrics; dependence-aware uncertainty; failure slices; computational context; and limits of interpretation.

**Completion exercise:** adapt the contract to a second public dataset or a new local capture. Explain any incompatibility instead of silently renaming its fields. Have another researcher review the target and split before training.

**Assessment:** the reader should be able to justify these choices, execute a baseline, diagnose an error and state the population supported by the result. Local notebook execution alone does not establish that readers have achieved those outcomes.

## 7. Adapt an unfamiliar CSV: predict before opening the solution
`data/teaching/adaptation_link.csv` contains the first 42 calendar days of UVA sensor04, expressed using a different schema. It is a public-data teaching extract from the same deployment, not an independent benchmark. Its JSON sidecar records source, checksum, units and license.

**Your task:** predict the received-hour SNR three hours ahead from six closed hours of RSSI/SNR. Before running the solution, write the column mapping, the label-availability time, the exclusion rule, a chronological split, and two simple references. Explain why `receptions` cannot supply a delivery denominator. Keep timestamps as UTC and retain missing hours.

**Checkpoint:** changing this exercise to +6 h requires changing both the target shift and its availability timestamp. A future target may be missing even when a forecast can be issued.

In [ ]:
raw_new=pd.read_csv(ROOT/'data/teaching/adaptation_link.csv')
schema=json.loads((ROOT/'data/teaching/adaptation_link.json').read_text())
display(raw_new.head());display(pd.Series(schema['fields']))
# Write your proposed mapping and contract here before reading the next cell.


### Worked solution: map fields and preserve elapsed time
The complete index is reconstructed before shifts. The link identifiers are checked and excluded from the feature vector. Six measurements do not necessarily mean six consecutive hours.

In [ ]:
frame=raw_new.rename(columns={'closed_utc':'time','received_rssi_dbm':'rssi','received_snr_db':'snr'})
frame['time']=pd.to_datetime(frame.time,utc=True)
assert frame[['receiver','node']].drop_duplicates().shape[0]==1
assert not frame.time.duplicated().any()
frame=frame.set_index('time').sort_index().asfreq('1h')
history_hours,horizon_hours=6,3
fit_end=frame.index[int(.6*len(frame))];test_start=frame.index[int(.8*len(frame))]
def construct_examples(observations):
    x=pd.DataFrame({f'{v}_lag{k}':observations[v].shift(k)
                    for v in ['rssi','snr'] for k in range(history_hours)})
    features=list(x.columns)
    x['truth']=observations.snr.shift(-horizon_hours)
    x['label_available_at']=x.index+pd.Timedelta(hours=horizon_hours)
    x=x.dropna()
    x['split']=np.where((x.index<fit_end)&(x.label_available_at<fit_end),'train',
               np.where((x.index>=fit_end)&(x.index<test_start)&(x.label_available_at<test_start),'validation',
               np.where(x.index>=test_start,'test','purged')))
    return x[x.split.ne('purged')],features
examples,feature_columns=construct_examples(frame)
display(examples.groupby('split').size().rename('eligible examples'))
assert examples.loc[examples.split.eq('train'),'label_available_at'].max()<fit_end
assert examples.loc[examples.split.eq('validation'),'label_available_at'].max()<test_start


### Fit, choose and assess
Only the fitting subset determines standardization and coefficients. Validation selects Ridge's penalty. The test evaluates the frozen selection and the same-hour cohort for both references. This exercise produces a local teaching result, not a model-family ranking.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
fit_rows=examples.split.eq('train');validation_rows=examples.split.eq('validation');test_rows=examples.split.eq('test')
options=[]
for penalty in [.1,1,10,100]:
    fitted=make_pipeline(StandardScaler(),Ridge(alpha=penalty)).fit(examples.loc[fit_rows,feature_columns],examples.loc[fit_rows,'truth'])
    error=mean_absolute_error(examples.loc[validation_rows,'truth'],fitted.predict(examples.loc[validation_rows,feature_columns]))
    options.append((error,penalty,fitted))
_,selected_penalty,selected_model=min(options,key=lambda z:z[0])
held=examples.loc[test_rows]
predictions={'persistence':held.snr_lag0.to_numpy(),
             f'{history_hours}-hour mean':held[[f'snr_lag{k}' for k in range(history_hours)]].mean(axis=1).to_numpy(),
             'Ridge':selected_model.predict(held[feature_columns])}
display(pd.Series({name:mean_absolute_error(held.truth,pred) for name,pred in predictions.items()},name='test MAE (dB)'))
print('Validation-selected penalty:',selected_penalty)


### Test the information boundary and transfer the reasoning
Perturbing future observations must leave already available training examples unchanged. This check can catch a noncausal feature transformation. It complements the timestamp checks; it cannot establish that the source clock is correct.

In [ ]:
changed=frame.copy();changed.loc[changed.index>=test_start,['rssi','snr']]+=1000
changed_examples,_=construct_examples(changed)
pd.testing.assert_frame_equal(examples.loc[fit_rows],changed_examples.loc[examples.index[fit_rows]])
print('Future perturbation leaves training examples unchanged.')


**Independent adaptation:** change the history to 12 h and the horizon to 6 h. Before execution, predict which counts can change and explain the new label time. Then repeat the run and report the actual counts and errors. Do not keep the better test result as a newly validated choice.

**Failure task:** suppose `closed_utc` is a local clock with an unknown timezone, or the requested target changes to delivery probability. State which assumption fails and what additional information is needed. Replacing missing targets with zero does not solve either problem.

**Completion evidence:** submit the mapped schema, contract, counts, baseline results, boundary check and a three-sentence conclusion. Record any assistance received.

Record conceptual reasoning, implementation, assistance and transfer separately. Executing the supplied solution is practice; completing the new contract without it is the transfer task.